# Adaptive Gauss-Kronrod check (Rust)

Start JupyterLab with `workshop run -- notebook`. Uses the `thole` facade crate from the working tree. After changing the Rust code, run the `:dep` cell again.

In [ ]:
:dep thole = { path = "/project/crates/thole" }

In [ ]:
use thole::core::Tolerance;
use thole::quad::{gauss_kronrod, gauss_kronrod_rule, Error, Options};

## 1. A first call

The Python binding runs this same code, so the bits must equal the `hex` printed by the Python notebook.

In [ ]:
let r = gauss_kronrod(f64::exp, 0.0, 1.0, &Options::default()).unwrap();
println!("{r:?}");
println!("error = {:e}", (r.value - (1.0_f64.exp() - 1.0)).abs());
println!("bits  = {:#018x}", r.value.to_bits());

## 2. The constants

The Kronrod rule is exact up to degree 22 and the Gauss rule up to degree 13, up to the rounding of the constants.

In [ ]:
let (nodes, kronrod, gauss) = gauss_kronrod_rule();
for k in [0, 13, 14, 22, 24] {
    let exact = if k % 2 == 0 { 2.0 / (k as f64 + 1.0) } else { 0.0 };
    let moment = |w: &[f64; 15]| nodes.iter().zip(w).map(|(x, w)| w * x.powi(k)).sum::<f64>() - exact;
    println!("x^{k:<2}  Kronrod error {:.1e}  Gauss error {:.1e}", moment(&kronrod), moment(&gauss));
}

## 3. Reference integrals (mpmath, 50 digits) at two tolerances

In [ ]:
let cases: [(&str, fn(f64) -> f64, f64, f64, f64); 4] = [
    ("exp", f64::exp, 0.0, 1.0, 1.7182818284590453),
    ("sqrt", f64::sqrt, 0.0, 1.0, 2.0 / 3.0),
    ("-log", |x| -x.ln(), 0.0, 1.0, 1.0),
    ("peak", |x| 1.0 / (1e-4 + (x - 0.3) * (x - 0.3)), 0.0, 1.0, 309.3986915124149),
];
let tight = Options { tol: Tolerance::new(1e-14, 1e-14), max_subintervals: 1000 };
for (name, f, a, b, expected) in cases {
    for opts in [Options::default(), tight] {
        let r = gauss_kronrod(f, a, b, &opts).unwrap();
        println!("{name:5} value={:.16} error={:.1e} estimate={:.1e} subintervals={}", r.value, (r.value - expected).abs(), r.error, r.subintervals);
    }
}

## 4. Failures

In [ ]:
let opts = Options::default();
let results: Vec<(&str, Result<_, Error>)> = vec![
    ("infinite end", gauss_kronrod(f64::exp, 0.0, f64::INFINITY, &opts)),
    ("zero limit", gauss_kronrod(f64::exp, 0.0, 1.0, &Options { max_subintervals: 0, ..opts })),
    ("f returns inf", gauss_kronrod(|x| 1.0 / x, -1.0, 1.0, &opts)),
    ("limit too small", gauss_kronrod(f64::sqrt, 0.0, 1.0, &Options { tol: Tolerance::new(1e-14, 0.0), max_subintervals: 3 })),
];
for (name, r) in &results {
    println!("{name:16} -> {r:?}");
}